# Bonus — Stacking et blending de modèles

**Jour 3 — chapitre 02 : Mise en pratique (aller plus loin)**

## Mise en situation

Combiner les prédictions de plusieurs modèles différents améliore souvent la performance
au-delà de chaque modèle pris isolément. À réserver après avoir consolidé une bonne
baseline individuelle (voir le notebook template du pipeline).


In [1]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
})[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

titanic = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True)
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

features = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "Sex_male"]
features = [c for c in features if c in titanic.columns]

X = titanic[features]
y = titanic["Survived"]

titanic.head()


,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,FamilySize,IsAlone
0,0,3,22.0,1,0,7.2500,True,False,True,2,0
1,1,1,38.0,1,0,71.2833,False,False,False,2,0
2,1,3,26.0,0,0,7.9250,False,False,True,1,1
3,1,1,35.0,1,0,53.1000,False,False,True,2,0
4,0,3,35.0,0,0,8.0500,True,False,True,1,1


## Suivi des expériences avec MLflow

On réutilise la même expérience que dans le notebook template (`OFDS - Jour 3 - Compétition du jour`) : blending et stacking viennent s'ajouter aux runs `baseline`, `random_forest`, `gradient_boosting`... déjà enregistrés, pour une comparaison complète.

Plutôt que de garder une trace manuelle des scores dans un carnet ou un tableur (voir la
check-list du jour 3 : « a-t-on conservé une trace des expériences menées ? »), on utilise
ici **MLflow** pour enregistrer automatiquement les paramètres et les métriques de chaque
essai.

Ceci suppose qu'un serveur de tracking MLflow tourne en local, lancé au préalable avec :

```bash
mlflow server --host 127.0.0.1 --port 5001 \
    --backend-store-uri sqlite:///mlflow_data/mlflow.db \
    --default-artifact-root ./mlflow_data/mlartifacts
```

L'interface est ensuite consultable dans un navigateur à l'adresse
http://127.0.0.1:5001. Voir `demos/README.md` pour le détail, et
`speech/ressources-techniques.md` pour son rôle dans la stack technique.

Si aucun serveur n'est joignable (par exemple sur Kaggle Notebooks ou Colab, qui ne
voient pas votre machine locale), remplacez la ligne `mlflow.set_tracking_uri(...)`
ci-dessous par `mlflow.set_tracking_uri("file:./mlruns")` : MLflow écrit alors ses runs
dans un simple dossier local, consultable plus tard avec `mlflow ui`.


In [2]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("OFDS - Jour 3 - Compétition du jour")


2026/09/13 17:24:31 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/mouaad/Formation/plb/PFDS/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


<Experiment: artifact_location='/private/tmp/claude-501/-Users-mouaad-Formation-plb-PFDS/1618c5d4-eac3-4b9b-8c92-386b5ee7255b/scratchpad/mlflow_validation/mlartifacts/4', creation_time=1789313061644, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1789313061644, lifecycle_stage='active', name='OFDS - Jour 3 - Compétition du jour', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele_logreg = LogisticRegression(max_iter=5000)
modele_rf = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)
modele_gb = GradientBoostingClassifier(random_state=42)

for nom, modele in [("Régression logistique", modele_logreg), ("Random Forest", modele_rf), ("Gradient Boosting", modele_gb)]:
    with mlflow.start_run(run_name="base_" + nom.lower().replace(" ", "_")):
        scores = cross_val_score(modele, X_train, y_train, cv=5)
        mlflow.log_param("model", type(modele).__name__)
        mlflow.log_metric("cv_accuracy_mean", scores.mean())
    print(f"{nom:22s} -> score moyen CV = {scores.mean():.3f}")


View run base_régression_logistique at: http://127.0.0.1:5001/#/experiments/4/runs/73666112cee042f098056cb42c718d96
View experiment at: http://127.0.0.1:5001/#/experiments/4
Régression logistique  -> score moyen CV = 0.803


View run base_random_forest at: http://127.0.0.1:5001/#/experiments/4/runs/bf252bc3bc2d4272902540e23eeb0863
View experiment at: http://127.0.0.1:5001/#/experiments/4
Random Forest          -> score moyen CV = 0.827


View run base_gradient_boosting at: http://127.0.0.1:5001/#/experiments/4/runs/d855228fbf424ff58df013982c056dc8
View experiment at: http://127.0.0.1:5001/#/experiments/4
Gradient Boosting      -> score moyen CV = 0.831


## Blending — moyenne des prédictions de plusieurs modèles entraînés indépendamment

**Question de réflexion :** si on moyenne simplement les probabilités prédites par trois
modèles indépendants, s'attend-on à un résultat plus stable qu'un modèle unique ? Pourquoi ?


In [4]:
from sklearn.metrics import accuracy_score

with mlflow.start_run(run_name="blending"):
    modele_logreg.fit(X_train, y_train)
    modele_rf.fit(X_train, y_train)
    modele_gb.fit(X_train, y_train)

    proba_logreg = modele_logreg.predict_proba(X_test)[:, 1]
    proba_rf = modele_rf.predict_proba(X_test)[:, 1]
    proba_gb = modele_gb.predict_proba(X_test)[:, 1]

    proba_blend = (proba_logreg + proba_rf + proba_gb) / 3
    predictions_blend = (proba_blend >= 0.5).astype(int)
    test_accuracy_blend = accuracy_score(y_test, predictions_blend)

    mlflow.log_param("model", "blending (logreg + rf + gb)")
    mlflow.log_metric("test_accuracy", test_accuracy_blend)

print("Précision du blending sur le jeu de test local :", round(test_accuracy_blend, 3))


View run blending at: http://127.0.0.1:5001/#/experiments/4/runs/162a63561f23418199fc306f5495186a
View experiment at: http://127.0.0.1:5001/#/experiments/4
Précision du blending sur le jeu de test local : 0.81


**Ce qu'on observe** : le blending est rapide à mettre en oeuvre (une simple moyenne de
probabilités) et lisse les erreurs propres à chaque modèle individuel, à condition que
ces modèles ne se trompent pas systématiquement sur les mêmes observations.


## Stacking — un méta-modèle apprend à combiner les prédictions des modèles de base

**Question de réflexion :** en quoi le stacking va-t-il plus loin que le blending ? Que
signifie « méta-modèle » ici ?


In [5]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[
        ("logreg", LogisticRegression(max_iter=5000)),
        ("rf", RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)),
        ("gb", GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=5000),
    cv=5,
)

with mlflow.start_run(run_name="stacking"):
    scores_stacking = cross_val_score(stacking, X_train, y_train, cv=5)
    mlflow.log_param("model", "stacking (logreg + rf + gb -> logreg)")
    mlflow.log_metric("cv_accuracy_mean", scores_stacking.mean())

print("Stacking — score moyen CV :", round(scores_stacking.mean(), 3))


View run stacking at: http://127.0.0.1:5001/#/experiments/4/runs/8cba7d0179a7430796f91da996670665
View experiment at: http://127.0.0.1:5001/#/experiments/4
Stacking — score moyen CV : 0.829


**Ce qu'on observe** : plutôt qu'une moyenne fixe (blending), le `final_estimator` (ici
une régression logistique) apprend lui-même le poids optimal à accorder à chaque modèle
de base, en s'appuyant sur une cross-validation interne (`cv=5`) pour éviter que le
méta-modèle ne triche en regardant des prédictions faites sur les données qui ont servi à
entraîner les modèles de base.

**À retenir pour le débriefing** : sur un dataset aussi petit et déjà bien modélisé que le
Titanic, le gain du stacking par rapport au meilleur modèle individuel reste souvent
minime — ces techniques prennent surtout tout leur sens sur des compétitions avec des
gains de classement mesurés au millième, comme celles proposées en vraie grandeur le jour
3.

Ouvrez http://127.0.0.1:5001, expérience « OFDS - Jour 3 - Compétition du jour » : les
runs `base_*`, `blending` et `stacking` de ce notebook s'ajoutent à ceux du template de
pipeline. C'est le moment de trier par métrique dans l'interface MLflow pour désigner,
tous runs confondus, la meilleure configuration de la journée.
